### This notebook is about learning how to use Dataset and Dataloader inside our NN pipeline

In [6]:
import pandas as pd
import numpy as np
import torch
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from torch.utils.data import Dataset, DataLoader

### ***Using Brest Cancer data with Dataset and DataLoader***
**Use of Dataset and DataLoader->**
- Dataset is used to load the data from the files or server into the system where we will run the model and evaluate.
    - It is used to create a complete pipeline where we can load, and preprocess the entire data all in one go.

- DataLoader is used to load the data into batches instead of using the entire data all at once for training the model.
    - It can also shuffle the data inside the pipeline itself, to create unbiased training data


In [4]:
df = pd.read_csv("breast_cancer.csv")
X = df.drop(['id','diagnosis','Unnamed: 32'],axis=1)
y = df['diagnosis']

In [5]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2, random_state=42)

In [7]:
# Standardize the data and LabelEncode the Output
scaler = StandardScaler()
encode = LabelEncoder()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train_enc = encode.fit_transform(y_train)
y_test_enc = encode.transform(y_test)

In [8]:
# Convert these into tensors of dtype float32
X_train_scaled = torch.from_numpy(X_train_scaled).to(dtype=torch.float32)
X_test_scaled = torch.from_numpy(X_test_scaled).to(dtype=torch.float32)
y_train_enc = torch.from_numpy(y_train_enc).to(dtype=torch.float32).view(-1,1)
t_test_enc = torch.from_numpy(y_test_enc).to(dtype=torch.float32).view(-1,1)

In [16]:
## Create dataset and dataloader to get the data
class CustomDataset(Dataset):
    def __init__(self, x_features , y_label):
        self.features = x_features
        self.labels = y_label

    def __len__(self):
        return len(self.features)

    def __getitem__(self , idx):
        return self.features[idx], self.labels[idx]

In [17]:
# forming the train and test datasets
train_dataset = CustomDataset(X_train_scaled , y_train_enc)
test_dataset = CustomDataset(X_test_scaled , y_test_enc) 

In [18]:
# forming the train and test dataloaders
train_loader = DataLoader(train_dataset , batch_size = 32 , shuffle=True)
test_loader = DataLoader(test_dataset , batch_size = 32 , shuffle=True)

In [19]:
# Creating the model
import torch.nn as nn
class MySimpleNN(nn.Module):

    def __init__(self , num_features):

        super().__init__()
        self.network = nn.Sequential( # building a simple neural net with only 1 layer
            nn.Linear(num_features , 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        out = self.network(features)
        return out

In [21]:
# create parameters
learning_rate = 0.1
epochs = 100

#create model
model = MySimpleNN(X_train_scaled.shape[1])

# create optimizer
optimizer = torch.optim.SGD(model.parameters() , lr = learning_rate)

# define the loss function
loss_function = nn.BCELoss()


In [22]:
## Now let us run the model and train it!!

for epoch in range(epochs):
    for batch_features, batch_labels in train_loader:

        # forward pass
        y_pred = model(batch_features)

        # loss calculate
        loss = loss_function(y_pred , batch_labels)

        # clear gradients before hand
        optimizer.zero_grad()

        # backpropagation
        loss.backward()

        # update the weights and bias using optimizer

        optimizer.step()

        print(f'Loss at epoch {epoch+1} -> {loss}')



Loss at epoch 1 -> 0.6874487996101379
Loss at epoch 1 -> 0.5450617671012878
Loss at epoch 1 -> 0.4669799208641052
Loss at epoch 1 -> 0.39642584323883057
Loss at epoch 1 -> 0.4298118054866791
Loss at epoch 1 -> 0.3458532691001892
Loss at epoch 1 -> 0.3120094835758209
Loss at epoch 1 -> 0.32154250144958496
Loss at epoch 1 -> 0.23344776034355164
Loss at epoch 1 -> 0.20133455097675323
Loss at epoch 1 -> 0.2467953860759735
Loss at epoch 1 -> 0.24751785397529602
Loss at epoch 1 -> 0.23600760102272034
Loss at epoch 1 -> 0.2337554395198822
Loss at epoch 1 -> 0.37310314178466797
Loss at epoch 2 -> 0.2562389075756073
Loss at epoch 2 -> 0.17957735061645508
Loss at epoch 2 -> 0.25452980399131775
Loss at epoch 2 -> 0.20425209403038025
Loss at epoch 2 -> 0.25010162591934204
Loss at epoch 2 -> 0.16212601959705353
Loss at epoch 2 -> 0.15383273363113403
Loss at epoch 2 -> 0.23740939795970917
Loss at epoch 2 -> 0.11052499711513519
Loss at epoch 2 -> 0.20888996124267578
Loss at epoch 2 -> 0.1900272667407

In [26]:
# Evaluation

model.eval() # this sets the model to evaluation mode
accuracy = []
with torch.no_grad():
    for batch_features , batch_labels in test_loader:
        # forward pass
        y_pred = model(batch_features)

        # convert probs to binary
        y_pred = (y_pred>0.8).float() 

        # check the accuracy and append it
        accuracy.append(accuracy_score(y_pred , batch_labels))

# final accuracy score

print(f' Final Accuracy Score -> {round(sum(accuracy)/len(accuracy),3)}')



 Final Accuracy Score -> 0.984
